In [ ]:
"""
BEV v5 — calibration-aware + rover-specialist model.

What is new vs v4:
1. Explicit rig features from intrinsics/extrinsics, not only rover_id
2. Camera-wise FiLM on image features from per-camera calibration
3. BEV-wise FiLM from global rig summary + rover embedding
4. Specialist conditioning for top test rovers (shared trunk, specialised residual prior)
5. Test-matched group split helper
"""
from __future__ import annotations

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from bev_v1 import (
    BEV_H,
    BEV_RES,
    BEV_W,
    CAR2CAM_NAMES,
    CAMERA_NAMES,
    GT_NAME,
    INTRINSICS_NAMES,
    SmallUNet,
    X_RANGE,
    Y_RANGE,
    Z_LEVELS,
)
from bev_v4 import BEVDatasetV4, build_rover_vocab


def _camera_pose_features(car2cam: np.ndarray) -> np.ndarray:
    """
    We mostly need stable rig geometry. Translation is the strongest signal,
    and camera forward axis gives a compact orientation hint.
    """
    cam2car = np.linalg.inv(car2cam)
    trans = cam2car[:3, 3].astype(np.float32)
    forward = cam2car[:3, 2].astype(np.float32)
    return np.concatenate([trans, forward], axis=0)


def build_rig_features(
    intrinsics: np.ndarray,
    car2cams: np.ndarray,
    img_hw: tuple[int, int],
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      cam_features: (N, 10)
      global_features: (12,)
    """
    H_t, W_t = img_hw
    cam_feats = []
    poses = []
    for K, M in zip(intrinsics, car2cams):
        pose = _camera_pose_features(M)
        poses.append(pose)
        cam_feat = np.array(
            [
                K[0, 0] / max(W_t, 1),   # fx norm
                K[1, 1] / max(H_t, 1),   # fy norm
                K[0, 2] / max(W_t, 1),   # cx norm
                K[1, 2] / max(H_t, 1),   # cy norm
                pose[0] / 10.0,
                pose[1] / 10.0,
                pose[2] / 10.0,
                pose[3],
                pose[4],
                pose[5],
            ],
            dtype=np.float32,
        )
        cam_feats.append(cam_feat)

    cam_feats = np.stack(cam_feats, axis=0)
    poses = np.stack(poses, axis=0)

    # Derived rig summary for the whole sample.
    left_pose = poses[2]
    right_pose = poses[3]
    mid_pose = poses[0]
    far_pose = poses[1]
    global_feat = np.array(
        [
            float(np.mean(cam_feats[:, 0])),                 # mean fx norm
            float(np.std(cam_feats[:, 0])),                  # std fx norm
            float(np.mean(cam_feats[:, 1])),                 # mean fy norm
            float(np.std(cam_feats[:, 1])),                  # std fy norm
            float(abs(left_pose[1] - right_pose[1]) / 10.0),  # lr baseline
            float(abs(mid_pose[0] - far_pose[0]) / 10.0),     # front delta x
            float(abs(mid_pose[2] - far_pose[2]) / 10.0),     # front delta z
            float(np.mean(poses[:, 0]) / 10.0),
            float(np.mean(poses[:, 1]) / 10.0),
            float(np.mean(poses[:, 2]) / 10.0),
            float(np.std(poses[:, 0]) / 10.0),
            float(np.std(poses[:, 1]) / 10.0),
        ],
        dtype=np.float32,
    )
    return cam_feats, global_feat


def build_specialist_vocab(
    train_info_csv: Path,
    test_info_csv: Path,
    min_train_count: int = 40,
    topk_test_rovers: int = 12,
) -> dict[str, int]:
    train_info = pd.read_csv(train_info_csv, index_col=0)
    test_info = pd.read_csv(test_info_csv, index_col=0)
    train_counts = Counter(train_info["rover"])
    test_counts = Counter(test_info["rover"])

    selected = []
    for rover, _ in test_counts.most_common():
        if train_counts.get(rover, 0) >= min_train_count:
            selected.append(rover)
        if len(selected) >= topk_test_rovers:
            break
    return {rover: i for i, rover in enumerate(selected)}


def make_test_matched_split(
    train_info_csv: Path,
    test_info_csv: Path,
    group_cols: tuple[str, str] = ("rover", "ride_date"),
    holdout_frac: float = 0.2,
    seed: int = 42,
    cache_path: Path | None = None,
) -> tuple[list[int], list[int]]:
    """
    Group holdout, but rover proportions are biased towards the test leaderboard mix.
    """
    if cache_path is not None and Path(cache_path).exists():
        d = np.load(cache_path)
        return d["train_idx"].tolist(), d["val_idx"].tolist()

    train_info = pd.read_csv(train_info_csv, index_col=0).reset_index(drop=True)
    test_info = pd.read_csv(test_info_csv, index_col=0).reset_index(drop=True)
    rng = np.random.RandomState(seed)

    test_rovers = set(test_info["rover"].unique())
    test_counts = Counter(test_info["rover"])
    test_total = sum(test_counts.values())

    rover_to_groups: dict[str, list[tuple]] = defaultdict(list)
    grouped = train_info.groupby(list(group_cols)).groups
    for key in grouped.keys():
        rover = key[0]
        if rover in test_rovers:
            rover_to_groups[rover].append(key)

    val_groups = set()
    for rover, groups in rover_to_groups.items():
        groups = list(groups)
        rng.shuffle(groups)
        base_frac = holdout_frac
        test_weight = test_counts[rover] / max(test_total, 1)
        # Slightly oversample validation groups for dominant test rovers.
        rover_frac = min(0.5, max(0.1, base_frac * (0.75 + 2.0 * test_weight)))
        n_holdout = max(1, int(round(len(groups) * rover_frac)))
        val_groups.update(groups[:n_holdout])

    train_idx, val_idx = [], []
    for i, row in train_info.iterrows():
        key = tuple(row[c] for c in group_cols)
        if key in val_groups:
            val_idx.append(i)
        else:
            train_idx.append(i)

    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(
            cache_path,
            train_idx=np.array(train_idx, dtype=np.int64),
            val_idx=np.array(val_idx, dtype=np.int64),
        )
    return train_idx, val_idx


class BEVDatasetV5(BEVDatasetV4):
    def __init__(
        self,
        data_dir: Path,
        mode: str = "train",
        img_hw=(384, 704),
        aug: bool = False,
        rover_vocab: dict | None = None,
        specialist_vocab: dict | None = None,
    ):
        super().__init__(
            data_dir=data_dir,
            mode=mode,
            img_hw=img_hw,
            aug=aug,
            rover_vocab=rover_vocab,
        )
        self.specialist_vocab = specialist_vocab or {}
        self._unknown_specialist = len(self.specialist_vocab)

    def _load_sample(self, idx: int):
        out = super()._load_sample(idx)
        row = self.info.iloc[idx]

        intrinsics_np = out["intrinsics"].numpy().astype(np.float32)
        car2cams_np = out["car2cams"].numpy().astype(np.float32)
        cam_rig, global_rig = build_rig_features(intrinsics_np, car2cams_np, self.img_hw)

        out["cam_rig"] = torch.from_numpy(cam_rig)         # (4, 10)
        out["global_rig"] = torch.from_numpy(global_rig)   # (12,)
        rover = row.get("rover", "?")
        spec_id = self.specialist_vocab.get(rover, self._unknown_specialist)
        out["specialist_id"] = torch.tensor(spec_id, dtype=torch.long)
        return out


class _ResNet34Backbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        rn = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(128, 128, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        return self.proj(x)


class _FiLM(nn.Module):
    def __init__(self, cond_dim: int, feat_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feat_dim * 2),
        )

    def forward(self, cond: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        return gamma, beta


class MultiCamBEVv5(nn.Module):
    def __init__(
        self,
        num_rovers: int,
        num_specialists: int,
        rover_emb_dim: int = 24,
        specialist_emb_dim: int = 16,
        n_cameras: int = 4,
        freeze_backbone: bool = False,
    ):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS

        self.backbone = _ResNet34Backbone(pretrained=True)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.cam_film = _FiLM(cond_dim=10, feat_dim=64)

        self.num_rovers = num_rovers + 1
        self.num_specialists = num_specialists + 1
        self.rover_embed = nn.Embedding(self.num_rovers, rover_emb_dim)
        self.specialist_embed = nn.Embedding(self.num_specialists, specialist_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        nn.init.normal_(self.specialist_embed.weight, std=0.02)

        bev_cond_dim = 12 + rover_emb_dim + specialist_emb_dim
        self.bev_film = _FiLM(cond_dim=bev_cond_dim, feat_dim=64 * len(self.z_levels), hidden_dim=96)
        self.specialist_gate = nn.Sequential(
            nn.Linear(bev_cond_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

        self.register_buffer("ego_voxels", self._make_ego_voxels(), persistent=False)
        self.bev_decoder = SmallUNet(
            in_c=64 * len(self.z_levels),
            base_c=32,
            out_c=1,
        )

    def _make_ego_voxels(self):
        xs = torch.linspace(
            self.x_range[0] + BEV_RES / 2,
            self.x_range[1] - BEV_RES / 2,
            self.bev_h,
        )
        ys = torch.linspace(
            self.y_range[0] + BEV_RES / 2,
            self.y_range[1] - BEV_RES / 2,
            self.bev_w,
        )
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing="ij")
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(
        self,
        images: torch.Tensor,
        intrinsics: torch.Tensor,
        car2cams: torch.Tensor,
        rover_ids: torch.Tensor | None = None,
        cam_rig: torch.Tensor | None = None,
        global_rig: torch.Tensor | None = None,
        specialist_ids: torch.Tensor | None = None,
    ):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        imgs_flat = images.reshape(B * N, C_, Hi, Wi)
        feat = self.backbone(imgs_flat)
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        if cam_rig is not None:
            cam_cond = cam_rig.reshape(B * N, -1)
            gamma, beta = self.cam_film(cam_cond)
            gamma = gamma.view(B, N, 64, 1, 1)
            beta = beta.view(B, N, 64, 1, 1)
            feat = feat * (1.0 + 0.1 * gamma) + 0.1 * beta

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum("bnij,bnvj->bniv", car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum("bnij,bnjv->bniv", intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        feat_flat = feat.reshape(B * N, 64, Hf, Wf)
        sampled = F.grid_sample(
            feat_flat,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        sum_v = sampled.sum(dim=1)
        cnt = valid_f.sum(dim=1).clamp(min=1.0)
        agg = sum_v / cnt
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        if rover_ids is None:
            rover_ids = torch.zeros(B, dtype=torch.long, device=agg.device)
        if specialist_ids is None:
            specialist_ids = torch.zeros(B, dtype=torch.long, device=agg.device)
        if global_rig is None:
            global_rig = torch.zeros(B, 12, dtype=agg.dtype, device=agg.device)

        rover_emb = self.rover_embed(rover_ids)
        specialist_emb = self.specialist_embed(specialist_ids)
        bev_cond = torch.cat([global_rig, rover_emb, specialist_emb], dim=-1)
        gamma, beta = self.bev_film(bev_cond)
        gamma = gamma.view(B, -1, 1, 1)
        beta = beta.view(B, -1, 1, 1)
        gate = self.specialist_gate(bev_cond).view(B, 1, 1, 1)
        agg = agg * (1.0 + 0.1 * gamma) + 0.1 * gate * beta

        return self.bev_decoder(agg)


def dump_v5_metadata(
    out_path: Path,
    rover_vocab: dict,
    specialist_vocab: dict,
    notes: dict | None = None,
):
    payload = {
        "rover_vocab_size": len(rover_vocab),
        "specialist_vocab_size": len(specialist_vocab),
        "specialist_rovers": specialist_vocab,
        "notes": notes or {},
    }
    out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))


Unknown instance spec: Please select VM configuration

# Eval / threshold sweep / submission

Универсальный eval-ноутбук для `v1`, `v2`, `v4`, `v5`.

Что умеет:
- честно грузить `model` или `ema` state dict;
- работать с `official`, `group`, `test_matched` split;
- делать streaming threshold sweep;
- генерировать submission zip.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, json, zipfile, hashlib
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import matplotlib.pyplot as plt

from bev_v1 import BEV_H, BEV_W
from bev_v3 import BEVDatasetAug, make_group_aware_split
from bev_v4 import BEVDatasetV4, build_rover_vocab
from bev_v5 import BEVDatasetV5, build_specialist_vocab, make_test_matched_split

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

Unknown instance spec: Please select VM configuration

## Config

In [ ]:
cfg = {
    'run_dir': './runs/v5',
    'which_ckpt': 'ema',        # 'best' | 'ema' | 'last'
    'model_type': 'v5',        # 'v1' | 'v2' | 'v4' | 'v5'
    'state_key': 'auto',       # 'auto' | 'model' | 'ema'
    'img_hw': (384, 704),
    'batch_size': 4,
    'num_workers': 4,
    'split_mode': 'test_matched',  # 'official' | 'group' | 'test_matched'
    'sweep_min': 0.20,
    'sweep_max': 0.70,
    'sweep_step': 0.02,
    'infer_threshold_override': None,
    'make_submission': False,
    'submission_name': None,
}

RUN_DIR = Path(cfg['run_dir'])
CKPT_PATH = RUN_DIR / f"{cfg['which_ckpt']}.pt"
assert CKPT_PATH.exists(), CKPT_PATH
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

## Model / dataset factory

In [ ]:
def make_model_and_dataset_factory(cfg):
    img_hw = cfg['img_hw']

    if cfg['model_type'] == 'v1':
        from bev_v1 import BEVDataset, MultiCamBEV
        model = MultiCamBEV(freeze_backbone=False)
        def make_dataset(data_dir, mode):
            return BEVDataset(data_dir, mode=mode, img_hw=img_hw)
        return model, make_dataset

    if cfg['model_type'] == 'v2':
        from bev_v2 import MultiCamBEVPretrainedEncoder
        model = MultiCamBEVPretrainedEncoder(load_pretrained=False, freeze_encoder=False)
        def make_dataset(data_dir, mode):
            return BEVDatasetAug(data_dir, mode=mode, img_hw=img_hw, aug=False)
        return model, make_dataset

    if cfg['model_type'] == 'v4':
        from bev_v4 import MultiCamBEVv4
        rover_vocab = build_rover_vocab(DATA_TRAIN/'info.csv', DATA_VAL/'info.csv', DATA_TEST/'info.csv')
        model = MultiCamBEVv4(num_rovers=len(rover_vocab))
        def make_dataset(data_dir, mode):
            return BEVDatasetV4(data_dir, mode=mode, img_hw=img_hw, aug=False, rover_vocab=rover_vocab)
        return model, make_dataset

    if cfg['model_type'] == 'v5':
        from bev_v5 import MultiCamBEVv5
        rover_vocab = build_rover_vocab(DATA_TRAIN/'info.csv', DATA_VAL/'info.csv', DATA_TEST/'info.csv')
        specialist_vocab = build_specialist_vocab(DATA_TRAIN/'info.csv', DATA_TEST/'info.csv')
        model = MultiCamBEVv5(num_rovers=len(rover_vocab), num_specialists=len(specialist_vocab))
        def make_dataset(data_dir, mode):
            return BEVDatasetV5(
                data_dir, mode=mode, img_hw=img_hw, aug=False,
                rover_vocab=rover_vocab, specialist_vocab=specialist_vocab,
            )
        return model, make_dataset

    raise ValueError(cfg['model_type'])


def resolve_state_dict(ckpt, which_ckpt='best', state_key='auto'):
    if state_key == 'model':
        return ckpt['model'], 'model'
    if state_key == 'ema':
        return ckpt['ema'], 'ema'
    if which_ckpt == 'ema' and 'ema' in ckpt:
        return ckpt['ema'], 'ema'
    return ckpt['model'], 'model'


def model_forward(model, batch, model_type):
    imgs = batch['images'].to(device, non_blocking=True)
    intr = batch['intrinsics'].to(device, non_blocking=True)
    c2c = batch['car2cams'].to(device, non_blocking=True)

    if model_type in ['v1', 'v2']:
        return model(imgs, intr, c2c)
    if model_type == 'v4':
        rov = batch['rover_id'].to(device, non_blocking=True)
        return model(imgs, intr, c2c, rov)

    rov = batch['rover_id'].to(device, non_blocking=True)
    cam_rig = batch['cam_rig'].to(device, non_blocking=True)
    global_rig = batch['global_rig'].to(device, non_blocking=True)
    specialist = batch['specialist_id'].to(device, non_blocking=True)
    return model(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)

## Load checkpoint

In [ ]:
model, make_dataset = make_model_and_dataset_factory(cfg)
ckpt = torch.load(CKPT_PATH, map_location=device)
state_dict, used_state_key = resolve_state_dict(
    ckpt,
    which_ckpt=cfg['which_ckpt'],
    state_key=cfg['state_key'],
)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()

print(json.dumps({
    'ckpt_path': str(CKPT_PATH),
    'used_state_key': used_state_key,
    'missing_keys': len(missing),
    'unexpected_keys': len(unexpected),
    'meta': {
        k: ckpt.get(k) for k in [
            'epoch', 'val_iou', 'val_iou_off',
            'ema_val_iou', 'ema_val_iou_off',
            'best_iou', 'best_ema_iou',
        ] if k in ckpt
    },
}, indent=2, default=str))

## Val loader

In [ ]:
if cfg['split_mode'] == 'official':
    val_ds = make_dataset(DATA_VAL, mode='val')
    split_name = 'official'
elif cfg['split_mode'] == 'group':
    _, val_idx = make_group_aware_split(
        DATA_TRAIN / 'info.csv',
        group_cols=('rover', 'ride_date'),
        holdout_frac=0.2,
        seed=42,
        cache_path=RUN_DIR / 'group_split.npz',
    )
    val_ds = Subset(make_dataset(DATA_TRAIN, mode='val'), val_idx)
    split_name = 'group'
else:
    _, val_idx = make_test_matched_split(
        DATA_TRAIN / 'info.csv',
        DATA_TEST / 'info.csv',
        holdout_frac=0.2,
        seed=42,
        cache_path=RUN_DIR / 'test_matched_split.npz',
    )
    val_ds = Subset(make_dataset(DATA_TRAIN, mode='val'), val_idx)
    split_name = 'test_matched'

val_loader = DataLoader(
    val_ds,
    batch_size=cfg['batch_size'],
    shuffle=False,
    num_workers=cfg['num_workers'],
    pin_memory=True,
)

print(split_name, len(val_ds), 'samples')

## Streaming threshold sweep

In [ ]:
@torch.no_grad()
def streaming_threshold_sweep(model, loader, thresholds, model_type, ignore_value=255):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)

    model.eval()
    for batch in tqdm(loader, desc='streaming sweep'):
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model_forward(model, batch, model_type).float()
        probs = torch.sigmoid(logits).cpu()
        gt = batch['gt']
        valid = (gt != ignore_value)
        gt_b = ((gt == 1) & valid).float()

        for ti, t in enumerate(thresholds):
            pred = ((probs > t) & valid).float()
            inter[ti] += (pred * gt_b).sum().item()
            union[ti] += ((pred + gt_b).clamp(0, 1)).sum().item()

    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


thresholds = np.arange(cfg['sweep_min'], cfg['sweep_max'] + 1e-9, cfg['sweep_step']).round(4).tolist()
iou_by_t = streaming_threshold_sweep(model, val_loader, thresholds, cfg['model_type'])
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print(f'best threshold: {best_t:.3f} | IoU={best_iou:.4f}')
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.3f}  iou={iou:.4f}{marker}')

In [ ]:
ts = list(iou_by_t.keys())
ious = [iou_by_t[t] for t in ts]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts, ious, marker='o')
ax.axvline(best_t, color='red', linestyle='--', alpha=0.5, label=f'best={best_t:.3f}')
ax.grid(); ax.legend(); ax.set_xlabel('threshold'); ax.set_ylabel('IoU')
plt.tight_layout(); plt.show()

## Test inference + submission

In [ ]:
infer_t = cfg['infer_threshold_override'] if cfg['infer_threshold_override'] is not None else best_t
print('using infer threshold:', infer_t)

if cfg['make_submission']:
    test_ds = make_dataset(DATA_TEST, mode='test')
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
    test_info = pd.read_csv(DATA_TEST / 'info.csv', index_col=0)
    out_dir = DATA_TEST / 'predicted_static_grids'
    out_dir.mkdir(exist_ok=True)

    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='test inference'):
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model_forward(model, batch, cfg['model_type']).float()
            preds = (torch.sigmoid(logits) > infer_t).cpu().numpy().astype(np.int32)
            idxs = batch['info_idx']
            for k, idx in enumerate(idxs):
                out_path = test_info.iloc[idx.item()]['predicted_occupancy_grid']
                np.save(out_path, preds[k].reshape(1, BEV_H, BEV_W))

    sub_name = cfg['submission_name'] or f"submission_{RUN_DIR.name}_{cfg['model_type']}_{cfg['which_ckpt']}.zip"
    sub_path = Path(sub_name)
    if sub_path.exists():
        sub_path.unlink()

    with zipfile.ZipFile(sub_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted((DATA_TEST / 'predicted_static_grids').glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(sub_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    h = hashlib.sha256()
    with open(sub_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)

    print(json.dumps({
        'submission': str(sub_path.resolve()),
        'size_mb': round(sub_path.stat().st_size / 1e6, 3),
        'sha256': h.hexdigest(),
    }, indent=2))
else:
    print('cfg[\'make_submission\'] = False, inference skipped')